In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import time

BASE_URL = "https://ardupilot.org/copter/docs/"
BASE_DOMAIN = urlparse(BASE_URL).netloc
BASE_PATH = urlparse(BASE_URL).path.rstrip("/")
MAX_WORKERS = 15

visited = set()
result = {}

def is_valid(url):
    parsed = urlparse(url)
    if parsed.netloc != BASE_DOMAIN:
        return False
    if not parsed.path.startswith(BASE_PATH):
        return False
    if url.lower().endswith((".pdf", ".zip", ".jpg", ".png", ".gif")):
        return False
    return True

def clean_url(url):
    url = url.split("#")[0]   # drop fragment
    url = url.split("?")[0]   # drop query string (e.g. ?C=D;O=A sort links)
    return url.rstrip("/")

def fetch_page(url):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
    except Exception as e:
        return url, None, str(e)

    soup = BeautifulSoup(resp.text, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        full_url = clean_url(urljoin(resp.url, a["href"]))
        if is_valid(full_url) and full_url != url:
            links.add(full_url)
    return url, links, None

def crawl(start_url):
    frontier = [clean_url(start_url)]
    visited.add(frontier[0])

    while frontier:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_page, url): url for url in frontier}
            next_frontier = []

            for future in as_completed(futures):
                url, links, error = future.result()

                if error:
                    result[url] = {"error": error}
                    print(f"Failed: {url} -> {error}")
                else:
                    result[url] = sorted(links)
                    print(f"Crawled: {url} -> {len(links)} links")
                    for link in links:
                        if link not in visited:
                            visited.add(link)
                            next_frontier.append(link)

                with open("links_result.json", "w") as f:
                    json.dump(result, f, indent=2)

        frontier = next_frontier

if __name__ == "__main__":
    start = time.time()
    crawl(BASE_URL)
    print(f"\nDone. Total pages crawled: {len(result)}")
    print(f"Time taken: {time.time() - start:.1f}s")
    print("Saved to links_result.json")

Crawled: https://ardupilot.org/copter/docs -> 916 links
Crawled: https://ardupilot.org/copter/docs/common-pxfmini.html -> 36 links
Crawled: https://ardupilot.org/copter/docs/common-generators.html -> 73 links
Crawled: https://ardupilot.org/copter/docs/common-compassless.html -> 101 links
Crawled: https://ardupilot.org/copter/docs/common-rpm.html -> 72 links
Crawled: https://ardupilot.org/copter/docs/common-ark-rtk-base.html -> 93 links
Crawled: https://ardupilot.org/copter/docs/common-ppm-encoders-new.html -> 68 links
Crawled: https://ardupilot.org/copter/docs/common-jae-jfb110.html -> 157 links
Crawled: https://ardupilot.org/copter/docs/common-ark-fpv.html -> 158 links
Crawled: https://ardupilot.org/copter/docs/acro-mode.html -> 48 links
Crawled: https://ardupilot.org/copter/docs/common-uavcan-peripherals.html -> 105 links
Crawled: https://ardupilot.org/copter/docs/channel-7-and-8-options.html -> 114 links
Crawled: https://ardupilot.org/copter/docs/ekf-inav-failsafe.html -> 55 links
C